In [6]:
import numpy as np

v2 = np.array([1,0])
v1 = np.array([0,1])
v3 = np.array([np.sqrt(2),np.sqrt(2)])
dot_product = np.dot(v1, v2)

print(v1.shape)
print(np.sqrt(np.sum(v1**v2)))
print(np.linalg.norm(v3))
print(v1 @ v3)

(2,)
1.0
2.0
1.4142135623730951




v1 and v2 are unit vectors because their magnitude, given by the arrow length, is one. v3 isn’t a unit vector, and its magnitude is two, twice the size of v1 and v2.

v1 and v2 are orthogonal because their tails meet at a 90 degree angle. You see this visually but can also verify it computationally by computing the dot product between v1 and v2. By using the dot product definition, v1 ⋅ v2 = ||v1|| ||v2|| cos(θ), you can see that when θ = 90, cos(θ) = 0 and v1 ⋅ v2 = 0. Intuitively, you can think of v1 and v2 as being totally unrelated or having nothing to do with each other. This will become important later.

v3 makes a 45 degree angle with both v1 and v2. This means that v3 will have a non-zero dot product with v1 and v2. This also means that v3 is equally related to both v1 and v2. In general, the smaller the angle between two vectors, the more they point toward a common direction.


## Vector Similarity

The ability to measure vector similarity is crucial in machine learning and mathematics more broadly. The foundation for this measurement lies in the dot product, which serves as the bedrock for many vector similarity metrics.

One issue with the dot product, when used in isolation, is that it can take on any value and is therefore difficult to interpret in absolute terms. For example, if you know only that the dot product between two vectors is -3, then it’s unclear what that means without more context.

To overcome this shortcoming, one common approach is to use cosine similarity, a normalized form of the dot product. You compute cosine similarity by taking the cosine of the angle between two vectors. In essence, you rearrange the cosine definition of the dot product from earlier to solve for cos(θ). The equation for cosine similarity looks like this:
Cosine Similarity Equation

![hello](https://realpython.com/cdn-cgi/image/width=982,format=auto/https://files.realpython.com/media/Screenshot_2023-10-30_at_1.17.14_PM.56871536fa90.png)

## Encode Objects in Embeddings

The next step in your journey to understanding and using vector databases like ChromaDB is to get a feel for embeddings. Embeddings are a way to represent data such as words, text, images, and audio in a numerical format that computational algorithms can more easily process.

More specifically, embeddings are dense vectors that characterize meaningful information about the objects that they encode. The most common kinds of embeddings are word and text embeddings, and that’s what you’ll focus on in this tutorial.
Word Embeddings

## Word Embeddings

A word embedding is a vector that captures the semantic meaning of word. Ideally, words that are semantically similar in natural language should have embeddings that are similar to each other in the encoded vector space. Analogously, words that are unrelated or opposite of one another should be further apart in the vector space.

One of the best ways to conceptualize this idea is to plot example word vectors in two dimensions. Take a good look at this scatterplot:

![image](https://realpython.com/cdn-cgi/image/width=1159,format=auto/https://files.realpython.com/media/Screenshot_2023-09-03_at_2.52.02_PM.5681ade2fa81.png)

```Note: If you’re interested in how word embeddings are created, then check out the Word2vec and GloVe algorithms. These algorithms create static word embeddings like the ones that you’ll use later in this section, but there are other ways to create dynamic embeddings. For example, the model underlying most large language models (LLMs), including ChatGPT, creates word embeddings that change based on the context surrounding the word.```

In [7]:
# spaCy library, a general-purpose NLP library

# also need to download a model that provides word embeddings, 
# among other features. For this tutorial, you’ll want to install 
# the medium or large English model:
# uv run python -m spacy download en_core_web_md

import spacy

# SpaCy’s en_core_web_md model includes 20,000 pre-trained word embeddings,
# each of which has 300 dimensions. This is more than enough for the examples
# that you’ll see next, but if you have the appetite for more word embeddings,
# then you can download the en_core_web_lg model, which has 514,000 embeddings.

nlp = spacy.load("en_core_web_md")
dog_embedding = nlp.vocab["dog"].vector

print(type(dog_embedding))
print(dog_embedding.shape)
print(dog_embedding[0:10])


<class 'numpy.ndarray'>
(300,)
[-0.72483   0.42538   0.025489 -0.39807   0.037463 -0.29811  -0.28279
  0.29333   0.57775   1.2205  ]


In [11]:
import numpy as np

def compute_cosine_similarity(u: np.ndarray, v: np.ndarray) -> float:
    """Compute the cosine similarity between two vectors"""
    return (u @ v) / (np.linalg.norm(u) * np.linalg.norm(v))

cat_embedding = nlp.vocab["cat"].vector
apple_embedding = nlp.vocab["apple"].vector
tasty_embedding = nlp.vocab["tasty"].vector
delicious_embedding = nlp.vocab["delicious"].vector
truck_embedding = nlp.vocab["truck"].vector

print(compute_cosine_similarity(cat_embedding, apple_embedding))
print(compute_cosine_similarity(cat_embedding, truck_embedding))
print(compute_cosine_similarity(tasty_embedding, delicious_embedding))
print(compute_cosine_similarity(truck_embedding, delicious_embedding))

0.2334378
0.18767197
0.450864
0.036047027


Word embeddings are great for capturing the semantic relationships between words, but what if you wanted to take things to the next level and analyze the similarity between sentences or documents? It turns out you accomplish this with text embeddings, and these are the kinds of embeddings that you’ll most often store in vector databases. More on that in the next section.

## Text Embeddings

Text embeddings encode information about sentences and documents, not just individual words, into vectors. This allows you to compare larger bodies of text to each other just like you did with word vectors. Because they encode more information than a single word embedding, text embeddings are a more powerful representation of information.

In [1]:
from sentence_transformers import SentenceTransformer

model =  SentenceTransformer("all-MiniLM-L6-v2")
texts = [
    "The canine barked loudly",
    "He ate a lot of pizzas",
]

text_embeding = model.encode(texts)
print(type(text_embeding))
print(text_embeding.shape)


d:\Dev\.personal\.uni-final\rag-finalYear-a1\.venv\Lib\site-packages\huggingface_hub\file_download.py:138: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\Joe\.cache\huggingface\hub\models--sentence-transformers--all-MiniLM-L6-v2. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
Loading weights: 100%|██████████| 103/103 [00:00<00:00, 2666.47it/s]


<class 'numpy.ndarray'>
(2, 384)
